# 04 · Multilingual LLMs & Cross-Lingual Transfer
### *Aligning & Deploying LLMs — Unit 2*

One model, many languages. In this notebook we make three multilingual ideas from the deck concrete:

1. **Tokenization imbalance** — the same sentence costs more tokens in some languages.
2. **A shared embedding space** — same meaning lands in the same place across languages.
3. **Evaluation** — why BLEU vs chrF matters, and cross-lingual transfer.

> CPU is fine.

In [ ]:
!pip -q install "transformers>=4.40" sentence-transformers sacrebleu matplotlib

## 1 · Tokenization imbalance ("fertility")

Subword vocabularies are trained mostly on English-heavy data, so other languages get chopped into **more tokens per
word**. More tokens = higher cost, more latency, and less room in the context window. We compare a multilingual
tokenizer's token counts on the same sentence across languages.

In [ ]:
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

tok = AutoTokenizer.from_pretrained("xlm-roberta-base")

sentences = {
    "English":  "Artificial intelligence is changing the world.",
    "French":   "L'intelligence artificielle change le monde.",
    "Hindi":    "कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।",
    "Chinese":  "人工智能正在改变世界。",
    "Arabic":   "الذكاء الاصطناعي يغير العالم.",
    "Telugu":   "కృత్రిమ మేధస్సు ప్రపంచాన్ని మారుస్తోంది.",
}

counts = {lang: len(tok.tokenize(s)) for lang, s in sentences.items()}
for lang, n in counts.items():
    print(f"{lang:9s} {n:2d} tokens   | {sentences[lang]}")

plt.figure(figsize=(8, 4))
plt.bar(counts.keys(), counts.values(), color="#6d5df0")
plt.ylabel("tokens for the same sentence"); plt.title("Tokenization cost by language (xlm-roberta)")
plt.tight_layout(); plt.show()

Notice how non-Latin scripts often need **2–4×** the tokens of English for the *same meaning*. This is a real,
recurring cost of multilingual deployment.

## 2 · A shared cross-lingual embedding space

A multilingual sentence encoder maps text from any language into one space where **meaning ≈ position**. We embed the
same three concepts in several languages and show the cosine-similarity matrix: same-meaning pairs light up
**regardless of language**.

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np, matplotlib.pyplot as plt

enc = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

texts = [
    "a cat sitting on a mat",      # EN
    "un chat assis sur un tapis",  # FR (cat)
    "एक बिल्ली चटाई पर बैठी है",     # HI (cat)
    "the stock market crashed today",      # EN
    "le marché boursier s'est effondré",   # FR (market)
    "आज शेयर बाजार गिर गया",                # HI (market)
]
labels = ["cat·EN", "cat·FR", "cat·HI", "market·EN", "market·FR", "market·HI"]

emb = enc.encode(texts, convert_to_tensor=True)
sim = util.cos_sim(emb, emb).cpu().numpy()

plt.figure(figsize=(6.5, 5.5))
plt.imshow(sim, cmap="viridis", vmin=0, vmax=1)
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, f"{sim[i,j]:.2f}", ha="center", va="center",
                 color="white" if sim[i,j] < 0.6 else "black", fontsize=8)
plt.title("Cross-lingual similarity (same meaning ≈ high, any language)")
plt.colorbar(); plt.tight_layout(); plt.show()

## 3 · Zero-shot cross-lingual transfer

Because meaning aligns across languages, a classifier built in **one** language works in **others** with no extra
labels. Here we classify sentences in several languages just by nearest **label embedding** — a stand-in for a
fine-tuned head — demonstrating transfer.

In [ ]:
label_texts = {"positive": "this is wonderful, I love it",
               "negative": "this is terrible, I hate it"}
label_emb = {k: enc.encode(v, convert_to_tensor=True) for k, v in label_texts.items()}

samples = [
    ("English",  "The food was absolutely delicious!"),
    ("French",   "Ce film était vraiment ennuyeux."),
    ("Hindi",    "यह अनुभव शानदार था।"),
    ("Spanish",  "El servicio fue horrible y lento."),
]
for lang, s in samples:
    e = enc.encode(s, convert_to_tensor=True)
    pred = max(label_emb, key=lambda k: util.cos_sim(e, label_emb[k]).item())
    print(f"{lang:8s} → {pred:8s} | {s}")

## 4 · Evaluation: BLEU vs chrF

Automatic MT metrics disagree, especially for morphology-rich or non-Latin languages. **chrF** works at the character
level and is fairer to such languages than word-level **BLEU**. Always pair automatic scores with human judgement.

In [ ]:
import sacrebleu

reference = ["The cat is sitting on the mat."]
hypotheses = {
    "good":     "The cat sits on the mat.",
    "paraphrase":"A cat is on the mat.",
    "poor":     "Dog runs in the park.",
}
for name, hyp in hypotheses.items():
    bleu = sacrebleu.corpus_bleu([hyp], [reference]).score
    chrf = sacrebleu.corpus_chrf([hyp], [reference]).score
    print(f"{name:11s}  BLEU {bleu:5.1f}   chrF {chrf:5.1f}   | {hyp}")

## Recap & your turn

- **mBERT / XLM-R** (encoder-only) and **XGLM** (decoder-only) share one vocabulary across languages.
- That shared space enables **zero-shot cross-lingual transfer** — train in English, run elsewhere.
- **Tokenization is uneven**: budget extra tokens (and cost) for non-Latin languages.
- Pick metrics per language: **chrF** often beats **BLEU**; verify with humans.

**Exercises**
1. Add Japanese and Russian to the fertility test — how do they compare?
2. Swap the encoder for `sentence-transformers/LaBSE` and re-run the similarity matrix.
3. Load `joeddav/xlm-roberta-large-xnli` and try true zero-shot classification with a `zero-shot-classification` pipeline.